<a href="https://colab.research.google.com/github/aminnademi/Clinical-Synthetic-Data-Audit/blob/main/pima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
pip install sdv

# Import Required Libraries

In [51]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import wasserstein_distance

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

import sdv
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer

In [52]:
SEED = 42
TEST_SIZE = 0.30
N_SYNTHETIC = 1000

OUT_DIR = Path("pima_benchmark_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Outcome"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed()

# Load Dataset & EDA

In [53]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]

df = pd.read_csv(url, header=None, names=columns)

In [54]:
df.shape

(768, 9)

In [55]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [56]:
display(df.describe())

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [57]:
df.isnull().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


### Data Separation

In [58]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df[TARGET],
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

### Handling Logical Missing Values

In [59]:
def prepare_data(data):
    X = data[FEATURES].copy()
    y = data[TARGET].astype(int).copy()

    # Replace invalid zeros with NaN
    for col in Zeros:
        X[col] = X[col].replace(0, np.nan)

    return X, y

Zeros = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI",]

FEATURES = df.columns

# Prepare train and test
X_train, y_train = prepare_data(train_df)
X_test, y_test = prepare_data(test_df)


# KNN imputation: fit only on train
imputer = KNNImputer(n_neighbors=5)

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=FEATURES,
    index=X_train.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=FEATURES,
    index=X_test.index
)


# Training data for the generators
train_model_df = X_train.copy()
train_model_df[TARGET] = y_train

### SDV for Metadata

In [60]:
metadata = SingleTableMetadata()

metadata.detect_from_dataframe(train_model_df)

metadata.update_column(
    column_name=TARGET,
    sdtype="categorical"
)

display(pd.DataFrame(
    metadata.to_dict()["columns"]
).T)

,sdtype
Pregnancies,numerical
Glucose,numerical
BloodPressure,numerical
SkinThickness,numerical
Insulin,numerical
BMI,numerical
DiabetesPedigreeFunction,numerical
Age,numerical
Outcome,categorical


### Train CTGAN & TVAE

In [63]:
CTGAN_EPOCHS = 300
TVAE_EPOCHS = 300

ctgan = CTGANSynthesizer(
    metadata=metadata,
    epochs=CTGAN_EPOCHS,
    enforce_min_max_values=False,
    enforce_rounding=True,
    verbose=False,
)

tvae = TVAESynthesizer(
    metadata=metadata,
    epochs=TVAE_EPOCHS,
    enforce_min_max_values=False,
    enforce_rounding=True,
    verbose=False,
)

print("[+] Training CTGAN...")
set_seed(SEED)
ctgan.fit(train_model_df)

print("[+] Training TVAE...")
set_seed(SEED)
tvae.fit(train_model_df)

print("[+] Finished.")

/usr/local/lib/python3.13/dist-packages/sdv/single_table/base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)


[+] Training CTGAN...
[+] Training TVAE...
[+] Finished.


### Generate 1000 Records

In [64]:
synthetic = {}

print("[+] Sampling CTGAN...")
set_seed(SEED)

synthetic["CTGAN"] = ctgan.sample(
    num_rows=N_SYNTHETIC
)

print("[+] Sampling TVAE...")
set_seed(SEED)

synthetic["TVAE"] = tvae.sample(
    num_rows=N_SYNTHETIC
)

for name in synthetic:
    data = synthetic[name].copy()

    # target را binary نگه می‌داریم
    data[TARGET] = (
        pd.to_numeric(
            data[TARGET],
            errors="coerce"
        )
        .round()
        .clip(0, 1)
        .astype(int)
    )

    synthetic[name] = data

    data.to_csv(
        OUT_DIR / f"{name.lower()}_raw.csv",
        index=False
    )

print({
    name: data.shape
    for name, data in synthetic.items()
})

[+] Sampling CTGAN...
[+] Sampling TVAE...
{'CTGAN': (1000, 9), 'TVAE': (1000, 9)}
